In [2]:
!pip install -q faster-whisper

from faster_whisper import WhisperModel

# "small" = fast + accurate enough. (use device="cpu", compute_type="int8" if no GPU)
whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Whisper loaded ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 29.4 MB/s eta 0:00:00


Whisper loaded ✅


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/kavach/kavach_muril"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device); model.eval()

def check_message(msg):
    enc = tokenizer(msg, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, -1)[0]
    v = "SCAM" if prob[1] > prob[0] else "SAFE"
    return f"{v} ({float(max(prob))*100:.1f}%)"
#

Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

In [4]:
def check_call(audio_path):
    # transcribe in the ORIGINAL language (task="transcribe", not "translate")
    segments, info = whisper_model.transcribe(audio_path, task="transcribe")
    text = " ".join(seg.text for seg in segments)

    print(f"🎙️ Detected language: {info.language}")
    print(f"📝 Transcript: {text[:250]}...")
    verdict = check_message(text)
    print(f"🛡️ VERDICT: {verdict}")
    return text, verdict

In [5]:
# upload an mp3/wav scam call recording to Colab, then:
check_call("/content/Recording (2).m4a")

🎙️ Detected language: en
📝 Transcript:  Thank you....
🛡️ VERDICT: SAFE (98.6%)


(' Thank you.', 'SAFE (98.6%)')

In [6]:
!pip install -q pydub

In [7]:
from pydub import AudioSegment

def stream_check_call(audio_path, chunk_sec=10):
    audio = AudioSegment.from_file(audio_path)
    total = len(audio) / 1000
    print(f"📞 Call length: {total:.0f}s — monitoring live in {chunk_sec}s chunks...\n")

    running_text = ""
    alerted = False
    for start in range(0, int(total), chunk_sec):
        # grab the next chunk of the call
        chunk = audio[start*1000:(start+chunk_sec)*1000]
        chunk.export("chunk.wav", format="wav")

        # transcribe just this chunk
        segments, info = whisper_model.transcribe("chunk.wav", task="transcribe")
        chunk_text = " ".join(seg.text for seg in segments).strip()
        running_text += " " + chunk_text

        # classify the conversation so far
        verdict = check_message(running_text)
        print(f"[{start:>3}-{start+chunk_sec:<3}s] {chunk_text[:55]}")
        print(f"            → {verdict}")

        # the moment it's a scam, alert immediately
        if "SCAM" in verdict and not alerted:
            print(f"\n🚨 SCAM DETECTED at {start+chunk_sec}s into the call!")
            print("   → Warning user + alerting Cyber Cell (before caller escapes)\n")
            alerted = True

    print("\n✅ Monitoring complete.")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [8]:
stream_check_call("/content/Scam Alert-1_ Fake Police Officer Tried to Extort ₹70,000 on WhatsApp! (Full Recorded Call)..mp3")

📞 Call length: 341s — monitoring live in 10s chunks...

[  0-10 s] जितने लड़कों के कारण आपका बेटा रेस्ट हुआ है ना उन पर तो
            → SCAM (99.6%)

🚨 SCAM DETECTED at 10s into the call!
   → Warning user + alerting Cyber Cell (before caller escapes)

[ 10-20 s] लड़कों को मैं बेज रहा हूं क्या करना भाईया नहीं नहीं नही
            → SCAM (99.6%)
[ 20-30 s] कि जो है प्लीज ऐसे मत करो तो प्लीज ऐसे मत करो कि भाईया 
            → SCAM (99.6%)
[ 30-40 s] प्लीज दो छोड़ दो प्लीज कि क्या करना है छोड़वाना
            → SCAM (99.6%)
[ 40-50 s] करना है कि क्या करना पड़ेगा कि भागर तक किसी खंचा पानी म
            → SCAM (99.6%)
[ 50-60 s] खर्चा पानी मैं तो भाईया गरीब आदमी हूँ  भाईया मेरी बात स
            → SCAM (99.6%)
[ 60-70 s] चाहते हैं कोई कैसे नहीं फुरवाना चाहेगा कि यह बोलेगर क्य
            → SCAM (99.6%)
[ 70-80 s] सर जी माफ कर दो सर जी कहां पकड़ा गया वैसे  पूले कहां आक
            → SCAM (99.6%)
[ 80-90 s] क्यों तो कहां तो कहां तो चुकाना पड़ेगा आज पूर्ण कर रहा 
            → SCAM (99.6%)
[ 

KeyboardInterrupt: 

Shriking the size of the model to load it on cpu